# Flood event mapping using Sentinel-1 GRD data

Map the extent of flooded areas using Sentinel-1 data for pre and post event.

Generate a pre-event mosaic and use the first available image after the event. Calculate the ratio between pre and post event status and apply a threshold to get the area affected by flooding. 

In [ ]:
import openeo
import rasterio
from openeo.processes import ProcessBuilder, array_element, quantiles, median
import rioxarray as rxr
import leafmap
from IPython.display import JSON
from shapely.geometry import shape
from folium.plugins import Draw
from scipy.ndimage import uniform_filter
import numpy as np

from pathlib import Path

In [23]:
connection = openeo.connect("https://openeo.dataspace.copernicus.eu")
connection.authenticate_oidc()

Authenticated using refresh token.


<Connection to 'https://openeo.dataspace.copernicus.eu/openeo/1.2/' with OidcBearerAuth>

In [3]:
# define properties for the outputs
from pathlib import Path

out_dir = Path("/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/flood/tuscany")
event = "Tuscany_2025_S1"

## 1) Define Area of Interest and dates of the event

In [10]:
# open a map and zoom to the area of interest
m = leafmap.Map(center=(46.65, 11.4), zoom=8.5)
m

Map(center=[46.65, 11.4], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

In [11]:
feat = m.draw_features
geom_dict = feat[0]['geometry']
geom = shape(geom_dict)

minx, miny, maxx, maxy = geom.bounds

bbox = {
    "west": minx,
    "south": miny,
    "east": maxx,
    "north": maxy,
}

print(bbox)

{'west': 10.658112, 'south': 43.696673, 'east': 10.947189, 'north': 43.877108}


In [12]:
PRE_DATE  = ("2025-01-01", "2025-03-10")   # before flood
POST_DATE = ("2025-03-15", "2025-03-20")   # during/after flood

## 2) Search stac catalog for items in the given time frame and area of interest

In [13]:
from pystac_client import Client
from datetime import datetime

catalog = Client.open("https://stac.dataspace.copernicus.eu/v1")

#for c in catalog.get_collections():
#    print(c.id)

In [14]:
# Search for Sentinel-1 GRD items
search = catalog.search(
    collections=["sentinel-1-grd"],   
    bbox=[bbox["west"], bbox["south"], bbox["east"], bbox["north"]],
    datetime=f"{PRE_DATE[0]}/{POST_DATE[1]}",
    
)
items = sorted(search.items(), key=lambda i: i.datetime)

In [15]:
print(items)

[<Item id=S1A_IW_GRDH_1SDV_20250104T052752_20250104T052817_057290_070C6E_1B14_COG>, <Item id=S1A_IW_GRDH_1SDV_20250105T171448_20250105T171513_057312_070D48_FD6E_COG>, <Item id=S1A_IW_GRDH_1SDV_20250111T051943_20250111T052008_057392_071074_9604_COG>, <Item id=S1A_IW_GRDH_1SDV_20250112T170627_20250112T170652_057414_071158_BC7E_COG>, <Item id=S1A_IW_GRDH_1SDV_20250116T052752_20250116T052817_057465_071357_25AA_COG>, <Item id=S1A_IW_GRDH_1SDV_20250117T171447_20250117T171512_057487_071437_3B5F_COG>, <Item id=S1A_IW_GRDH_1SDV_20250123T051942_20250123T052007_057567_071767_26BB_COG>, <Item id=S1A_IW_GRDH_1SDV_20250124T170626_20250124T170651_057589_07184D_FAAB_COG>, <Item id=S1A_IW_GRDH_1SDV_20250128T052751_20250128T052816_057640_071A53_922F_COG>, <Item id=S1A_IW_GRDH_1SDV_20250129T171447_20250129T171512_057662_071B2C_375E_COG>, <Item id=S1A_IW_GRDH_1SDV_20250204T051942_20250204T052007_057742_071E51_3405_COG>, <Item id=S1A_IW_GRDH_1SDV_20250205T170626_20250205T170651_057764_071F35_3630_COG>, <It

## 3) Search for S1 GRD dates for pre and post event with matching orbit

In [ ]:

from datetime import datetime, timezone

event = datetime(2025, 3, 14, tzinfo=timezone.utc)

items = sorted(items, key=lambda i: i.datetime)

# Split before and after event
pre_candidates = [
    i for i in items 
    if i.datetime < event
]

post_candidates = [
    i for i in items 
    if i.datetime > event
]

# Find a matching orbit with enough pre-event images
orbit_pair = None

for post in sorted(post_candidates, key=lambda i: i.datetime):
    #orbit_state = post.properties["sat:orbit_state"]
    relative_orbit = post.properties["sat:relative_orbit"]

    matching_pre = [
        pre for pre in pre_candidates
        if (
            #pre.properties["sat:orbit_state"] == orbit_state
            pre.properties["sat:relative_orbit"] == relative_orbit
        )
    ]

    if len(matching_pre) >= 2:  # require multiple images
        orbit_pair = (matching_pre, post)
        break

if orbit_pair is None:
    raise RuntimeError("No matching Sentinel-1 orbit with enough pre-event images found.")

pre_images, post = orbit_pair

# Sort pre-images chronologically
pre_images = sorted(pre_images, key=lambda i: i.datetime)

print("Selected pre-event images:")
for p in pre_images:
    print(p.datetime, p.properties["sat:orbit_state"], p.properties["sat:relative_orbit"])

print("\nSelected post-event image:")
print(post.datetime, post.properties["sat:orbit_state"], post.properties["sat:relative_orbit"])

Selected pre-event images:
2025-01-04 05:27:52.913896+00:00 descending 168
2025-01-16 05:27:52.031701+00:00 descending 168
2025-01-28 05:27:51.528541+00:00 descending 168
2025-02-09 05:27:51.119810+00:00 descending 168
2025-02-21 05:27:50.505862+00:00 descending 168
2025-03-05 05:27:50.496201+00:00 descending 168

Selected post-event image:
2025-03-17 05:27:50.792128+00:00 descending 168


## 4) Define final date window for pre event and post event 

In [17]:
pre_date = [pre_images[0].datetime.date(), pre_images[-1].datetime.date()]
pre_date = (pre_date[0].strftime("%Y-%m-%d"), pre_date[1].strftime("%Y-%m-%d"))
print(f"Pre-event date range: {pre_date} ")

post_date = (
    post.datetime.strftime("%Y-%m-%d"),
    post.datetime.strftime("%Y-%m-%d")
)

print(f"Post-event date: {post_date} ")

Pre-event date range: ('2025-01-04', '2025-03-05') 
Post-event date: ('2025-03-17', '2025-03-17') 


In [ ]:
#info = connection.describe_collection("SENTINEL1_GRD")
#print(info)

## 5) Load both data cubes

In [ ]:
lee_udf = openeo.UDF("""
import numpy as np
import xarray as xr
from scipy.ndimage import uniform_filter

def lee(img, size, enl):
    img = img.astype(np.float32)
    valid = np.isfinite(img)
    filled = np.where(valid, img, 0.0)
    n = np.maximum(uniform_filter(valid.astype(np.float32), size, mode="reflect"), 1e-6)

    mean = uniform_filter(filled, size, mode="reflect") / n
    mean_sq = uniform_filter(filled ** 2, size, mode="reflect") / n
    var = np.maximum(mean_sq - mean ** 2, 0)

    noise_var = (mean ** 2) / enl                      # multiplicative speckle model
    w = np.maximum(var - noise_var, 0) / (var + 1e-10)

    out = mean + w * (img - mean)
    return np.where(valid, out, np.nan)

def apply_datacube(cube: xr.DataArray, context: dict) -> xr.DataArray:
    size = context.get("size", 7)
    enl = context.get("enl", 4.4)
    out = xr.apply_ufunc(
        lee, cube,
        kwargs=dict(size=size, enl=enl),
        input_core_dims=[["y", "x"]],
        output_core_dims=[["y", "x"]],
        vectorize=True,
    )
    return out.transpose(*cube.dims)
""")


def load_s1(temporal_extent):
    cube = connection.load_collection(
        "SENTINEL1_GRD",
        spatial_extent=bbox,
        temporal_extent=list(temporal_extent),
        bands=["VV", "VH"],
    ).sar_backscatter(coefficient="sigma0-ellipsoid")   # linear power, terrain-corrected
    return cube

def despeckle(cube, size=7, enl=4.4):
    return cube.apply_neighborhood(
        process=lee_udf,
        size=[{"dimension": "x", "value": 256, "unit": "px"},
              {"dimension": "y", "value": 256, "unit": "px"}],
        overlap=[{"dimension": "x", "value": 8, "unit": "px"},
                 {"dimension": "y", "value": 8, "unit": "px"}],
        context={"size": size, "enl": enl},
    )


def s1_composite(temporal_extent):
    filtered = despeckle(load_s1(temporal_extent))
    return filtered.reduce_dimension(dimension="t", reducer=mean)   # still linear


pre_s1  = s1_composite(PRE_DATE)
post_s1 = s1_composite(POST_DATE)

# Log-ratio: PRE/POST in dB 
to_db = lambda c: c.apply(lambda x: 10 * x.log(base=10))
ratio_db = to_db(pre_s1).merge_cubes(to_db(post_s1), overlap_resolver="subtract")

ratio_db.create_job(title=f"S1_logratio_{event}", out_format="GTiff") \
        .start_and_wait().download_results(out_dir)

### 7) Apply threshold to the result t0 get only the flooded areas

In [ ]:
thresh = 0.3

change = (ratio_db >= thresh).apply(lambda x: x * 1) 

job = change.create_job(title=f"S1_change_{event}", out_format="GTiff")
job.start_and_wait().download_results(out_dir)